In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import RobustScaler
from catboost import CatBoostRegressor
from IPython.display import display
import warnings
import sys

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Increase recursion limit for deep ensemble training
sys.setrecursionlimit(2000)

# ===========================
# Configuration
# ===========================

TRAIN = "/kaggle/input/house-prices-advanced-regression-techniques/train.csv"
TEST = "/kaggle/input/house-prices-advanced-regression-techniques/test.csv"
OPTUNA_TRIALS_CATBOOST = 100 # Trials for CatBoost hyperparameter tuning
OPTUNA_TRIALS_BLENDING = 50  # Trials for ensemble weight optimization


# ===========================
# Data Preparation Functions (All 5 Feature Engineering Ideas Included)
# ===========================

# --- Imputation Helpers ---
def nan_report(df: pd.DataFrame, stage: str = ""):
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if not missing.empty:
        print(f"\n{stage} → {len(missing)} columns with missing values")

def impute_column(col: pd.Series, strategy: str = "median"):
    cnt = col.isna().sum()
    if cnt == 0: return col
    if strategy == "median":
        return col.fillna(col.median())
    else:
        return col.fillna(0)

def impute_missing_values(df: pd.DataFrame):
    cat_none = ['Alley','PoolQC','Fence','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','GarageType','GarageFinish','GarageQual','GarageCond','MasVnrType','MSZoning','Utilities','Exterior1st','Exterior2nd','KitchenQual','SaleType','FireplaceQu','MiscFeature']
    num_zero = ['GarageArea','GarageCars','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','BsmtFullBath','BsmtHalfBath','MasVnrArea']
    for c in cat_none:
        if c in df.columns: df[c] = df[c].fillna('None')
    for c in num_zero:
        if c in df.columns: df[c] = df[c].fillna(0)
    if 'LotFrontage' in df.columns and 'Neighborhood' in df.columns:
        df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(lambda grp: impute_column(grp, strategy="median"))
    if 'GarageYrBlt' in df.columns and 'YearBuilt' in df.columns:
        df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['YearBuilt'])
    return df

def feature_engineering(df: pd.DataFrame):
    # 1. Total Square Footage (TotalSF)
    if all(col in df.columns for col in ['TotalBsmtSF', '1stFlrSF', '2ndFlrSF']):
        df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
        
    # 3. Age and Renovation Timeliness
    if 'YrSold' in df.columns:
        if 'YearBuilt' in df.columns: df['Age'] = (df['YrSold'] - df['YearBuilt']).apply(lambda x: max(0, x))
        if 'YearRemodAdd' in df.columns: df['RemodelAge'] = (df['YrSold'] - df['YearRemodAdd']).apply(lambda x: max(0, x))
        if 'GarageYrBlt' in df.columns: df['GarageAge'] = (df['YrSold'] - df['GarageYrBlt']).apply(lambda x: max(0, x))
             
    # 2. Quality-Area Interaction
    if all(col in df.columns for col in ['OverallQual', 'TotalSF']):
        df['OverallQual_x_TotalSF'] = df['OverallQual'] * df['TotalSF']
        
    # 5. Binary Presence Indicators
    presence_map = {'2ndFlrSF': 'Has2ndFloor', 'PoolArea': 'HasPool', 'MiscVal': 'HasMiscFeatureValue', 
                    'Fireplaces': 'HasFireplace', 'TotalBsmtSF': 'HasBasement'}
    for original_col, new_col in presence_map.items():
        if original_col in df.columns:
            df[new_col] = (df[original_col] > 0).astype(int)
    
    return df

def simplify_rare_categories(df: pd.DataFrame, threshold: int = 10):
    # 4. Categorical Feature Simplification
    cat_cols = df.select_dtypes(include='object').columns
    for col in cat_cols:
        counts = df[col].value_counts()
        rare_categories = counts[counts < threshold].index
        if not rare_categories.empty:
            df[col] = df[col].replace(rare_categories, 'Rare_Cat')
    return df

def safe_numeric_fill(df: pd.DataFrame) -> pd.DataFrame:
    df_num = df.select_dtypes(include=np.number)
    return df_num.fillna(0)

# ==================================================
# OPTUNA OBJECTIVE FUNCTIONS
# ==================================================

def catboost_objective(trial, X_train_40, y_train_40, X_tune_20, y_tune_20):
    """Objective for CatBoost hyperparameter tuning (using 40% train and 20% tune sets)."""
    params = {
        'iterations': trial.suggest_int('iterations', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-6, 1.0, log=True),
        'random_seed': 42,
        'loss_function': 'RMSE',
        'verbose': 0,
    }
    
    model = CatBoostRegressor(**params)
    # Fit on 40% train, evaluate on 20% tune
    model.fit(X_train_40, y_train_40, eval_set=[(X_tune_20, y_tune_20)], early_stopping_rounds=100)
    
    preds = model.predict(X_tune_20)
    rmse = np.sqrt(mean_squared_error(y_tune_20, preds))
    return rmse

def ensemble_objective(trial, lasso_blend_preds, catboost_blend_preds, y_blend):
    """Objective for ensemble blending weight optimization (using 20% blend set)."""
    
    w_lasso = trial.suggest_float('w_lasso', 0.0, 1.0, step=0.01)
    w_catboost = 1.0 - w_lasso
    
    blended_preds = (w_lasso * lasso_blend_preds) + (w_catboost * catboost_blend_preds)
    rmse = np.sqrt(mean_squared_error(y_blend, blended_preds))
    
    return rmse

# ===========================
# Main Pipeline Execution
# ===========================

# 1. Load and Prepare Data
train = pd.read_csv(TRAIN)
test = pd.read_csv(TEST)
y_log = np.log1p(train["SalePrice"])
train_ID = train["Id"]
test_ID = test["Id"]
train = train.drop(["Id", "SalePrice"], axis=1)
test = test.drop("Id", axis=1)

train = impute_missing_values(train)
test = impute_missing_values(test)
train = feature_engineering(train)
test = feature_engineering(test)
full = pd.concat([train, test], axis=0)
full = simplify_rare_categories(full, threshold=10)
full = pd.get_dummies(full)
X_train_clean = full.iloc[:len(train), :].copy()
X_test_final = safe_numeric_fill(full.iloc[len(train):, :].copy())
X_full_train = safe_numeric_fill(X_train_clean).values
y_full_train = y_log.values


# 2. Implement 40/20/20/20 Split
# Total data split into [40% Train, 60% Remainder]
X_train_40, X_remainder_60, y_train_40, y_remainder_60 = train_test_split(
    X_full_train, y_full_train, test_size=0.6, random_state=42
)

# Remainder (60%) split into [20% Tune, 20% Blend, 20% Holdout]
# Tune (20%) is 1/3 of the Remainder (60%)
X_tune_20, X_rest_40, y_tune_20, y_rest_40 = train_test_split(
    X_remainder_60, y_remainder_60, test_size=0.6666, random_state=42
)

# The Rest (40%) split into [20% Blend, 20% Holdout]
X_blend_20, X_holdout_20, y_blend_20, y_holdout_20 = train_test_split(
    X_rest_40, y_rest_40, test_size=0.5, random_state=42
)

print("\n--- Data Split (40% Train / 20% Tune / 20% Blend / 20% Holdout) ---")
print(f"1. Training Set (40%): {X_train_40.shape[0]} samples")
print(f"2. Tuning Set (20%): {X_tune_20.shape[0]} samples (For CatBoost HP tuning)")
print(f"3. Blending Set (20%): {X_blend_20.shape[0]} samples (For Ensemble Weight optimization)")
print(f"4. Holdout Set (20%): {X_holdout_20.shape[0]} samples (Final, honest validation)")


# 3. Train Base Model: Lasso (Fit on 40% Train)
print("\n--- 3. Training Lasso Base Model ---")
lasso_scaler = RobustScaler()
X_train_40_scaled = lasso_scaler.fit_transform(X_train_40)
X_blend_20_scaled = lasso_scaler.transform(X_blend_20)

lasso_model = Lasso(alpha=0.0005, random_state=42, max_iter=10000)
lasso_model.fit(X_train_40_scaled, y_train_40)
lasso_blend_preds = lasso_model.predict(X_blend_20_scaled)
print(f"Lasso Base Blend RMSE (on 20% Blend Set): {np.sqrt(mean_squared_error(y_blend_20, lasso_blend_preds)):.5f}")


# 4. Tune Base Model: CatBoost (Fit on 40% Train, Evaluate on 20% Tune)
print(f"\n--- 4. Running Optuna CatBoost Tuning on 20% Tune Set ({OPTUNA_TRIALS_CATBOOST} trials) ---")

cat_study = optuna.create_study(direction='minimize')
cat_study.optimize(
    lambda trial: catboost_objective(trial, X_train_40, y_train_40, X_tune_20, y_tune_20), 
    n_trials=OPTUNA_TRIALS_CATBOOST,
)

best_cat_params = cat_study.best_params
print(f"Optimal CatBoost RMSE (on 20% Tune Set): {cat_study.best_value:.5f}")

# Train the optimal CatBoost model on 40% train data
optimal_catboost_model = CatBoostRegressor(**best_cat_params, random_seed=42, loss_function='RMSE', verbose=0)
optimal_catboost_model.fit(X_train_40, y_train_40)
# Generate predictions for blending on 20% blend set
catboost_blend_preds = optimal_catboost_model.predict(X_blend_20)
print(f"Optimized CatBoost Blend RMSE (on 20% Blend Set): {np.sqrt(mean_squared_error(y_blend_20, catboost_blend_preds)):.5f}")


# 5. Optimize Ensemble Blend Weights (Evaluate on 20% Blend Set)
print(f"\n--- 5. Running Optuna Ensemble Blending on 20% Blend Set ({OPTUNA_TRIALS_BLENDING} trials) ---")

blend_study = optuna.create_study(direction='minimize')
blend_study.optimize(
    lambda trial: ensemble_objective(trial, lasso_blend_preds, catboost_blend_preds, y_blend_20), 
    n_trials=OPTUNA_TRIALS_BLENDING
)

# Extract Optimal Weights
best_blend_params = blend_study.best_params
w_lasso_optimal = best_blend_params['w_lasso']
w_catboost_optimal = 1.0 - w_lasso_optimal

print("\n" + "="*50)
print("✅ Final Blending Weights Found")
print(f"Best Blended RMSE (on 20% Blend Set): {blend_study.best_value:.5f}")
print(f"Lasso Weight (w_Lasso): {w_lasso_optimal:.3f}")
print(f"CatBoost Weight (w_CatBoost): {w_catboost_optimal:.3f}")
print("="*50)

# ... (Sections 1 through 5 are correct and executed)

# 6. Generate Final Submission (Retrain on Full Data)
print("\n--- 6. Generating Final Submission (Retraining on 100% of Training Data) ---")

# Retrain Lasso on 100% of training data
X_full_train_scaled = lasso_scaler.fit_transform(X_full_train)
X_full_test_scaled = lasso_scaler.transform(X_test_final.values)
lasso_model.fit(X_full_train_scaled, y_full_train)
lasso_test_preds = lasso_model.predict(X_full_test_scaled)

# **FIX APPLIED HERE**
# Create a modifiable copy of the best CatBoost parameters
final_cat_params = best_cat_params.copy()

# Retrain Optimal CatBoost on 100% of training data
# We slightly increase iterations and reduce learning rate for the final full train run
final_cat_params['iterations'] = int(final_cat_params['iterations'] * 1.5) # Increase iterations
final_cat_params['learning_rate'] = final_cat_params['learning_rate'] * 0.7 # Decrease learning rate

optimal_catboost_model_final = CatBoostRegressor(
    **final_cat_params, # Unpack the modified parameters here
    random_seed=42, 
    loss_function='RMSE', 
    verbose=0
)
optimal_catboost_model_final.fit(X_full_train, y_full_train)
catboost_test_preds = optimal_catboost_model_final.predict(X_test_final.values)

# Apply the optimal blend weights to the test set predictions
preds_log_blended = (w_lasso_optimal * lasso_test_preds) + (w_catboost_optimal * catboost_test_preds)
preds_blended = np.expm1(preds_log_blended)

# Create Submission File
submission = pd.DataFrame({
    "Id": test_ID,
    "SalePrice": preds_blended
})
submission['SalePrice'] = np.clip(submission['SalePrice'], 0, None)
submission.to_csv("final_optimized_blended_submission.csv", index=False)
print("\nFinal Optimized Blended Submission done ✅")

# Create Submission File
submission = pd.DataFrame({
    "Id": test_ID,
    "SalePrice": preds_blended
})
submission['SalePrice'] = np.clip(submission['SalePrice'], 0, None)
submission.to_csv("final_optimized_blended_submission.csv", index=False)
print("\nFinal Optimized Blended Submission done ✅")
